# Sujet 7 — Retrouver l'épicentre d'une maladie neurodégénérative
Master 2 Mathématiques, UNIKIN — Projet de travaux dirigés (Prof. C. D. Bope), année 2025–2026.

Ce notebook reproduit **tous les calculs, figures et tableaux** du rapport. Il s'exécute de bout en bout dans un noyau SageMath.
- Partie A : calcul exact et symbolique (SageMath).
- Partie B : simulations NumPy, chargées par `load('tp7.py')` et `load('sujet7.py')`.

In [ ]:
# Versions (à reporter dans le rapport, section 4)
import numpy, scipy, matplotlib
print(version()); print("NumPy", numpy.__version__, "| SciPy", scipy.__version__, "| Matplotlib", matplotlib.__version__)

## A1. Spectres exacts de petits graphes (exercice 7.1) et graphe des deux cliques (exercice 7.3)

In [ ]:
for nom, G in [("chemin P6", graphs.PathGraph(6)), ("cycle C6", graphs.CycleGraph(6)), ("complet K6", graphs.CompleteGraph(6))]:
    L = G.laplacian_matrix()
    print(nom, ":", L.charpoly().factor(), "| mu2 =", sorted(L.eigenvalues())[1])

In [ ]:
G = graphs.CompleteGraph(5).disjoint_union(graphs.CompleteGraph(5))
G.relabel()                      # sommets 0..9 : 0..4 = première clique, 5..9 = seconde
G.add_edge(0, 5)                 # une seule arête entre les cliques
L = G.laplacian_matrix()
print(L.charpoly().factor())     # attendu : x (x - 5)^7 (x^2 - 7x + 2)
mu = sorted(L.eigenvalues()); print("mu2 =", mu[1], "≈", RDF(mu[1]))

## A2. Vérification symbolique de la formule de l'atrophie cumulée (exercice 7.3, question 2)
Forme équivalente : $a(t) = t\,\frac{\mathbb 1^\top x_0}{n}\mathbb 1 + \frac1\beta L^+(I-e^{-\beta L t})x_0$, avec $L^+=(L+J/n)^{-1}-J/n$.

In [ ]:
var('beta t s', domain='positive')
L3 = graphs.PathGraph(3).laplacian_matrix().change_ring(SR)
n3 = 3; J = matrix(SR, n3, n3, [1]*9)
x0 = vector(SR, [1, 0, 0])
E = (-beta*s*L3).exp()
a_int = vector([integrate(f, s, 0, t) for f in E*x0])            # définition a(t) = ∫ x(s) ds
Lplus = (L3 + J/n3).inverse() - J/n3                                # pseudo-inverse de Moore-Penrose
a_spec = t*(J/n3)*x0 + Lplus*(identity_matrix(SR, 3) - E.subs(s=t))*x0/beta
print([ (u - v).full_simplify() for u, v in zip(a_int, a_spec)])  # attendu : [0, 0, 0]

## A3. Ordre d'arrivée dans Fisher–KPP (Proposition 7.8, point 3)
Dérivées successives en $t=0$ sur le chemin pondéré 0–1–2–3, graine $x(0)=\varepsilon e_0$ : on attend $x_j^{(k)}(0)=0$ pour $k<j$ et $x_j^{(j)}(0)=\varepsilon\beta^j w_1\cdots w_j$.

In [ ]:
var('alpha epsilon w1 w2 w3')
Y = vector(SR, var('y0 y1 y2 y3'))
Wp = matrix(SR, [[0,w1,0,0],[w1,0,w2,0],[0,w2,0,w3],[0,0,w3,0]])
Lw = diagonal_matrix(SR, [sum(r) for r in Wp.rows()]) - Wp
F = -beta*Lw*Y + alpha*vector(SR, [z*(1 - z) for z in Y])
lie = lambda g: sum(diff(g, Y[i])*F[i] for i in range(4))      # dérivation le long du champ
init = {y0: epsilon, y1: 0, y2: 0, y3: 0}
for j in range(4):
    g, ders = Y[j], []
    for k in range(j + 1):
        ders.append(g.subs(init).factor()); g = lie(g).expand()
    print("x_%d :" % j, ders)

## A4. Automorphismes et graines indiscernables (Théorème 3.5 du rapport)

In [ ]:
A = G.automorphism_group()
print("ordre de Aut(G) =", A.order())          # attendu 1152 = 2 * 4! * 4!
print("orbites :", A.orbits())
# automorphisme sigma qui échange les deux cliques : i <-> i+5
sigma = {i: (i + 5) % 10 for i in range(10)}
assert all(G.has_edge(sigma[u], sigma[v]) for u, v in G.edges(labels=False))
P = matrix(QQ, 10, 10, lambda i, j: 1 if i == sigma[j] else 0)
print("P L = L P :", P*L == L*P)
H = block_matrix(QQ, [[identity_matrix(5), identity_matrix(5)]])/2   # moyenne des régions homologues
print("H P = H :", H*P == H)
# conséquence : H a(e_1) = H a(e_6) exactement, alors que a(e_1) != a(e_6)
Ln = L.change_ring(RDF)
Et = (-2*Ln).exp()        # beta = 1, t = 2 : vérification sur l'état x(t)
e1 = vector(RDF, 10, {1: 1}); e6 = vector(RDF, 10, {6: 1})
print("|x(e6) - P x(e1)| =", (Et*e6 - P.change_ring(RDF)*Et*e1).norm())
print("|H x(e1) - H x(e6)| =", (H.change_ring(RDF)*(Et*e1 - Et*e6)).norm(), "| |x(e1)-x(e6)| =", (Et*(e1 - e6)).norm())

## A5. Petit graphe sans automorphisme non trivial (contre-exemple)

In [ ]:
for H2 in graphs(6):
    if H2.is_connected() and H2.automorphism_group().order() == 1:
        print("graphe asymétrique à 6 sommets :", H2.edges(labels=False)); break

## B. Simulations NumPy (toutes les figures et tous les tableaux)
Les fichiers `.py` sont lus comme du Python pur. La variable `_SOUS_SAGE` empêche le bloc principal de `sujet7.py` de se lancer au chargement.

In [ ]:
_SOUS_SAGE = True
load('tp7.py')       # code de départ du cours (affiche le spectre et l'épicentre retrouvé)
load('sujet7.py')    # simulations du sujet 7

In [ ]:
# Validation du code : version vectorisée = epicentre() du cours
import numpy as np
W, mod = connectome_modulaire(); L80 = laplacien(W)
x0 = np.zeros(80); x0[3] = 1
a = atrophie(L80, x0, t=2.0) + np.random.default_rng(6).normal(0, 0.02, 80)
s1, S1 = epicentre(L80, a, (1.0, 2.0, 4.0)); s2, S2 = epicentre_diffusion(spectre(L80), a, (1.0, 2.0, 4.0))
print(s1, s2, np.abs(S1 - S2).max())

In [ ]:
import time
t0 = time.time()
out = figures_toutes()        # figures/fig1 ... fig7, environ 5 minutes
print("temps de calcul :", round(time.time() - t0, 1), "s"); print(out['chrono'])

In [ ]:
# Tableaux 2 et 3 du rapport
np.set_printoptions(precision=2, suppress=True)
for gen, mdl in [('diffusion', 'diffusion'), ('fkpp', 'diffusion'), ('fkpp', 'fkpp')]:
    print(gen, '->', mdl, '\n région exacte :\n', tableau(out['res'], gen, mdl, out['ps'], out['sigmas'], 0),
          '\n bon module :\n', tableau(out['res'], gen, mdl, out['ps'], out['sigmas'], 1))
for gen in ('diffusion', 'fkpp'):
    print('arêtes déplacées,', gen, '\n', tableau(out['res_ar'], gen, gen, out['ps'], out['sigmas'], 0))

In [ ]:
# Tableau 4 : extension bayésienne
for sig, d in out['bayes'].items(): print(sig, d)

In [ ]:
# Tableau 3 : estimateur naïf (région la plus atrophiée)
print(reference_naive())